In [23]:
# %pip install tensorflow tensorflow-hub

# import tensorflow as tf
# from tensorflow.keras import layers, models, optimizers, callbacks
# from pathlib import Path
# import numpy as np
# import matplotlib.pyplot as plt
# import os

%pip install tensorflow matplotlib scikit-learn

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns


Note: you may need to restart the kernel to use updated packages.


In [24]:
# # Diagnostic: Check what data paths exist
# from pathlib import Path

# PROJECT_DIR = Path.cwd().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd()
# print(f"PROJECT_DIR: {PROJECT_DIR}")
# print(f"\nContents of PROJECT_DIR:")
# for item in PROJECT_DIR.iterdir():
#     print(f"  {item.name}" + (" (dir)" if item.is_dir() else ""))

# # Check for dataset folders
# data_candidates = [
#     PROJECT_DIR / "data" / "classification _dataset",
#     PROJECT_DIR / "data" / "classification _dataset",
#     PROJECT_DIR / "data",
# ]
# print("\nChecking possible dataset locations:")
# for path in data_candidates:
#     exists = path.exists()
#     print(f"  {path}: {exists}")
#     if exists and path.is_dir():
#         try:
#             subdirs = [d.name for d in path.iterdir() if d.is_dir()]
#             print(f"    -> subdirectories: {subdirs}")
#         except:
#             pass
PROJECT_DIR = Path.cwd().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "classification_dataset"
MODELS_DIR = PROJECT_DIR / "models" / "custom_cnn"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224,224,3)
BATCH_SIZE = 32
EPOCHS = 30
SEED = 42


In [26]:
# PROJECT_DIR = Path.cwd().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd()
# DATA_DIR = PROJECT_DIR / "data" / "classification _dataset"
# MODELS_DIR = PROJECT_DIR / "models" / "transfer_learning"
# MODELS_DIR.mkdir(parents=True, exist_ok=True)

# IMG_SIZE = (224,224)
# BATCH_SIZE = 32
# EPOCHS = 25
# NUM_CLASSES = len([d for d in (DATA_DIR/"train").iterdir() if d.is_dir()])
# SEED = 42

# from tensorflow.keras.preprocessing import image_dataset_from_directory
# train_ds = image_dataset_from_directory(DATA_DIR/"train", image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED)
# val_ds = image_dataset_from_directory(DATA_DIR/"valid", image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)
# test_ds = image_dataset_from_directory(DATA_DIR/"test", image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

# normalization = tf.keras.layers.Rescaling(1./255)
# train_ds = train_ds.map(lambda x,y: (normalization(x), y)).cache().prefetch(tf.data.AUTOTUNE)
# val_ds = val_ds.map(lambda x,y: (normalization(x), y)).cache().prefetch(tf.data.AUTOTUNE)
# test_ds = test_ds.map(lambda x,y: (normalization(x), y)).cache().prefetch(tf.data.AUTOTUNE)

from tensorflow.keras.preprocessing import image_dataset_from_directory
train_ds = image_dataset_from_directory(DATA_DIR / "train", image_size=IMG_SIZE[:2], batch_size=BATCH_SIZE, seed=SEED)
val_ds = image_dataset_from_directory(DATA_DIR / "valid", image_size=IMG_SIZE[:2], batch_size=BATCH_SIZE, shuffle=False)
test_ds = image_dataset_from_directory(DATA_DIR / "test", image_size=IMG_SIZE[:2], batch_size=BATCH_SIZE, shuffle=False)

# Extract class_names before applying transformations
class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes:", class_names)

normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x,y: (normalization_layer(x), y)).cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x,y: (normalization_layer(x), y)).cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(lambda x,y: (normalization_layer(x), y)).cache().prefetch(tf.data.AUTOTUNE)


Found 20 files belonging to 2 classes.


Found 6 files belonging to 2 classes.
Found 4 files belonging to 2 classes.
Found 4 files belonging to 2 classes.
Classes: ['bird', 'drone']
Classes: ['bird', 'drone']


In [27]:
# from tensorflow.keras.applications import EfficientNetB0

# def build_tl_model(img_size=IMG_SIZE+(3,), num_classes=NUM_CLASSES, base_trainable=False):
#     base_model = EfficientNetB0(weights="imagenet", include_top=False, input_shape=img_size)
#     base_model.trainable = base_trainable
#     inputs = layers.Input(shape=img_size)
#     x = base_model(inputs, training=False)
#     x = layers.GlobalAveragePooling2D()(x)
#     x = layers.Dropout(0.3)(x)
#     x = layers.Dense(256, activation="relu")(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.Dropout(0.4)(x)
#     outputs = layers.Dense(num_classes, activation="softmax")(x)
#     model = models.Model(inputs, outputs)
#     return model

# model_tl = build_tl_model()
# model_tl.summary()

def build_custom_cnn(input_shape=IMG_SIZE, num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="custom_cnn")
    return model

model = build_custom_cnn()
model.summary()



Model: "custom_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,426 (435.26 KB)

 Trainable params: 110,722 (432.51 KB)

 Non-trainable params: 704 (2.75 KB)

In [33]:
LR = 1e-3
model.compile(
    optimizer=optimizers.Adam(learning_rate=LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint_cb = callbacks.ModelCheckpoint(str(MODELS_DIR/"best_model.h5"), save_best_only=True, monitor="val_accuracy", mode="max")
es_cb = callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
csv_logger = callbacks.CSVLogger(str(MODELS_DIR/"training_log.csv"))

print("Model compiled successfully!")
print(f"Learning Rate: {LR}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")


Model compiled successfully!
Learning Rate: 0.001
Batch Size: 32
Epochs: 30


In [ ]:
# history_head = model_tl.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=[ckpt, es, reduce_lr])
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint_cb, es_cb, reduce_lr, csv_logger]
)

# save history
import pickle
with open(MODELS_DIR/"history.pkl","wb") as f:
    pickle.dump(history.history, f)


Epoch 1/30


In [31]:
import matplotlib.pyplot as plt
hist = history.history

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(hist['loss'], label='train_loss')
plt.plot(hist['val_loss'], label='val_loss')
plt.legend(); plt.title("Loss")

plt.subplot(1,2,2)
plt.plot(hist['accuracy'], label='train_acc')
plt.plot(hist['val_accuracy'], label='val_acc')
plt.legend(); plt.title("Accuracy")
plt.show()


NameError: name 'history' is not defined